In [ ]:
import os
import sys
import json
import re

import numpy as np
import pandas as pd
from collections import defaultdict

import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize

In [ ]:
# Parent directory PATH
sys.path.append("..")

from utils.json import load_json, save_json

In [ ]:
def json_structure(data, max_depth=5, current_depth=0, path="root"):
    """
    Recursively explore and print the structure of JSON data up to a specified depth
    
    Parameters:
        data: The JSON data to explore
        max_depth: Maximum depth to explore
        current_depth: Current depth in the recursion

    Args:
        data: The JSON data to explore
        max_depth: Maximum depth to explore
        current_depth: Current depth in the recursion
        path: Current path in the JSON structure
    """
    
    # Indentation for visualizing depth
    indent = (4*" ") * current_depth
    
    if current_depth >= max_depth:
        print(f"{indent}[Reached max depth at {path}]")
        return
    
    # Handle dictionary
    if isinstance(data, dict):
        print(f"{indent}{path} (dict with {len(data)} keys)")
        for key, value in data.items():
            new_path = f"{path}.{key}" if path != "root" else key
            json_structure(value, max_depth, current_depth + 1, new_path)
    
    # Handle list
    elif isinstance(data, list):
        print(f"{indent}{path} (list with {len(data)} items)")
        if data and current_depth < max_depth - 1:
            # Show structure of first item as an example
            sample_item = data[0]
            new_path = f"{path}[0]"
            json_structure(sample_item, max_depth, current_depth + 1, new_path)
            
            # If there are multiple different structures in the list, show another example
            if len(data) > 1 and not all(type(item) == type(sample_item) for item in data):
                different_type_item = next((item for item in data if type(item) != type(sample_item)), None)
                if different_type_item:
                    new_path = f"{path}[different_type]"
                    json_structure(different_type_item, max_depth, current_depth + 1, new_path)

    # Handle other types
    elif isinstance(data, (str, int, float, bool, type(None))):
        # For primitive types, show a sample value
        if isinstance(data, str) and len(data) > 30:
            sample_value = f"{data[:30]}..." 
        else:
            sample_value = data
        print(f"{indent}{path} ({type(data).__name__}): {sample_value}")

In [ ]:
def json_keys(data):
    """
    Extract the key structural components of the dataset
    
    Parameters:
        data (list): The JSON data to explore

    Returns:
        dict: Dictionary containing the structural information
    """
    structure = {
        "main_keys": set(),
        "data_keys": set(),
        "prediction_keys": set(),
        "result_keys": set(),
        "value_keys": set(),
        "label_types": set()
    }
    
    # Process documents to extract generic structure
    for doc in data[:15]:
        # Main level keys
        for key in doc.keys():
            structure["main_keys"].add(key)
        
        # Data level keys
        if "data" in doc:
            for key in doc["data"].keys():
                structure["data_keys"].add(key)
        
        # Prediction level
        if "predictions" in doc and doc["predictions"]:
            for pred in doc["predictions"]:
                for key in pred.keys():
                    structure["prediction_keys"].add(key)
                
                # Result level
                if "result" in pred:
                    for res in pred["result"]:
                        for key in res.keys():
                            structure["result_keys"].add(key)
                        
                        # Value level and labels
                        if "value" in res:
                            for key in res["value"].keys():
                                structure["value_keys"].add(key)
                            
                            # Collect label types
                            if "labels" in res["value"]:
                                for label in res["value"]["labels"]:
                                    structure["label_types"].add(label)
    
    # Convert sets to sorted lists for nicer output
    for key in structure:
        structure[key] = sorted(list(structure[key]))
    
    return structure

In [ ]:
def extract_annotations(data):
    """
    Extract all annotations/predictions from the documents to a Dataframe
    
    Parameters:
        data (list): List of document JSON objects

    Returns:
        pd.DataFrame: DataFrame containing all annotations
    """

    # Dataframe new rows
    rows = [] 
    
    # Iterate through all documents
    for doc_index, doc, in enumerate(data):
        doc_id = doc["data"]["id"]
        text = doc["data"]["text"]

        # Empty row for non predicted 
        if "predictions" not in doc or not doc["predictions"]:
            row = {
                "doc_index": doc_index, # Represent the position of each document in your data array
                "doc_id": doc_id,
                "result_id": None,  # No result ID since there are no predictions
                "start": None,      # No start position
                "end": None,        # No end position
                "label": "No Prediction",  # Special label to indicate absence
                "text": None,       # No specific text segment
                "context": text[:50] + "..." if len(text) > 50 else text  # First 50 chars for context
            }
            rows.append(row)
            continue 
        
        # Process predictions
        for pred_index, pred in enumerate(doc["predictions"]):
            if "result" in pred:
                for res_index, res in enumerate(pred["result"]):
                    if "value" in res and "labels" in res["value"]:
                        start = res["value"]["start"]
                        end = res["value"]["end"]
                        
                        # Extract the labeled text segment
                        segment_text = text[start:end]
                        
                        # Create a row for each label
                        for label in res["value"]["labels"]:
                            row = {
                                "doc_index": doc_index,
                                "doc_id": doc_id,
                                "result_id": res["id"],
                                "start": start,
                                "end": end,
                                "label": label,
                                "text":segment_text,
                            }
                            rows.append(row)

                        # Sort rows by start position
                        rows = sorted(rows, key=lambda x: x["start"])
                        rows = sorted(rows, key=lambda x: x["doc_index"])
    
    return pd.DataFrame(rows)

In [ ]:
# Load the training data
path = "../data/raw/negacio_train_v2024.json"
data = load_json(path)

In [ ]:
print(f"Type of loaded data: {type(data)}")
print(f"Number of documents: {len(data)}")

In [ ]:
# Explore your JSON data
json_structure(data, max_depth=10)

In [ ]:
# Get the JSON keys structure
structure = json_keys(data)

display(structure)

In [ ]:
# Insights using the first document
record_first = data[0]

if isinstance(data, list):
    print(f"Type document: {type(data[0])}")
    print(f"Keys document: {structure['main_keys']}")
    print()

    for main_keys in record_first:
        print(f"Key: {main_keys}")
        print(f"Type: {type(record_first[main_keys])}")
        print(f"Length: {len(record_first[main_keys])}")
        print(f"Value: {record_first[main_keys]}")
        print()

        if "data" in main_keys:
            for k in record_first["data"].keys():
                print(f"\tKey: {k}")
                print(f"\tType: {type(record_first['data'][k])}")
                print(f"\tLength: {len(record_first['data'][k])}")
                print(f"\tValue: {record_first['data'][k]}")
                print()

In [ ]:
# Extract all annotations
df_predictions = extract_annotations(data)

print(f"Extracted {len(df_predictions)} annotations")
display(df_predictions)

In [ ]:
# Annotation counts per document
annotation_counts = df_predictions.groupby("doc_id").size().sort_values(ascending=False)

print()
print("Top 5 documents by annotation count:")
for doc_id, count in annotation_counts.head(10).items():
    print(f"Document {doc_id} has {count} annotations")

In [ ]:
# Duplicated documents
true_counts = {}

# Count each unique document ID only once
for doc in data:
    doc_id = doc["data"]["id"]
    if doc_id in true_counts:
        true_counts[doc_id] += 1
    else:
        true_counts[doc_id] = 1

# Find actually duplicated IDs
actual_duplicates = {id: count for id, count in true_counts.items() if count > 1}

print(f"Number of documents: {len(data)}")
print(f"Number of unique document IDs: {len(true_counts)}")
print(f"Number of truly duplicated IDs: {len(actual_duplicates)}")
print()

if actual_duplicates:
    print("True duplicate document IDs:")
    for doc_id, count in actual_duplicates.items():
        print(f"ID {doc_id} appears {count} times")
else:
    print("No duplicate document IDs found")

In [ ]:
# Count the occurrences of each label
label_counts = df_predictions["label"].value_counts()

print("Label Distribution:")
for label, count in label_counts.items():
    print(f"{label}: {count} ({count/len(df_predictions)*100:.2f}%)")

labels = label_counts.index
counts = label_counts.values

# Plot the distribution using Matplotlib directly
plt.figure(figsize=(12, 7))
plt.bar(labels, counts, alpha=0.75)
plt.title("Distribution of Label Types")
plt.xlabel("Count")
plt.ylabel("Label")
plt.tight_layout()
plt.show()

In [ ]:
# Common Negation Cues
neg_cues = df_predictions[df_predictions["label"] == "NEG"]["text"].value_counts().head(20)
print("TOP Negation Cues: \n")
for cue, count in neg_cues.items():
    print(f"{cue}: {count}")

In [ ]:
# Common Uncertainty Cues
unc_cues = df_predictions[df_predictions["label"] == "UNC"]["text"].value_counts().head(20)
print("TOP Uncertainty Cues: \n")
for cue, count in unc_cues.items():
    print(f"{cue}: {count}")

In [ ]:
# Calculate scope lengths
df_predictions["text_length"] = df_predictions["text"].apply(len)

scope_labels = ["NSCO", "USCO"]

# Average Scope's 
for label in scope_labels:
    avg_len = df_predictions[df_predictions["label"] == label]["text_length"].mean()
    print(f"Average {label} length: {avg_len:.2f} characters")

# Plot scope length distributions by label type
plt.figure(figsize=(12, 7))
for label in scope_labels:
    scope_lengths = df_predictions[df_predictions["label"] == label]["text_length"]
    plt.hist(scope_lengths, bins=50, alpha=0.75, label=label)

plt.title("Distribution of Scope Lengths")
plt.xlabel("Length in Characters")
plt.ylabel("Frequency")
plt.legend()
plt.xlim(0, 100) # Limit to most common
plt.tight_layout()
plt.show()

### Create groups of Cues-Scope
We agrupate them based on distances in the text. A scope is assigned to the closest cue.

In [ ]:
# create a new column to store the scope for a CUE
df_predictions["scope"] = None

for index, row in df_predictions.iterrows():
    if row["label"] == "NSCO":
        before = df_predictions.iloc[index - 1]["label"] if index > 0 else None
        after = df_predictions.iloc[index + 1]["label"] if index < len(df_predictions) - 1 else None
        if before == "NEG":
            if df_predictions.iloc[index - 1]["end"] == row["start"]:
                df_predictions.at[index, "scope"] = index - 1
        elif after == "NEG":
            if df_predictions.iloc[index + 1]["start"] == row["end"]:
                df_predictions.at[index, "scope"] = index + 1
        else:
            df_predictions.at[index, "scope"] = None

    elif row["label"] == "USCO":
        before = df_predictions.iloc[index - 1]["label"] if index > 0 else None
        after = df_predictions.iloc[index + 1]["label"] if index < len(df_predictions) - 1 else None
        if before == "UNC":
            if df_predictions.iloc[index - 1]["end"] == row["start"]:
                df_predictions.at[index, "scope"] = index - 1
        elif after == "UNC":
            if df_predictions.iloc[index + 1]["start"] == row["end"]:
                df_predictions.at[index, "scope"] = index + 1
        else:
            df_predictions.at[index, "scope"] = None

# second pass
for index, row in df_predictions.iterrows():
    if row["label"] == "NSCO":
        if row["scope"] == None:
            before = df_predictions.iloc[index - 1]["label"] if index > 0 else None
            after = df_predictions.iloc[index + 1]["label"] if index < len(df_predictions) - 1 else None

            if before == "NEG":
                df_predictions.at[index, "scope"] = index - 1
            elif after == "NEG":
                df_predictions.at[index, "scope"] = index + 1

    elif row["label"] == "USCO":
        if row["scope"] == None:
            before = df_predictions.iloc[index - 1]["label"] if index > 0 else None
            after = df_predictions.iloc[index + 1]["label"] if index < len(df_predictions) - 1 else None

            if before == "UNC":
                df_predictions.at[index, "scope"] = index - 1
            elif after == "UNC":
                df_predictions.at[index, "scope"] = index + 1